In [ ]:
# 1. INSTALL
!pip install -q --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu118
!pip install -q implicit scikit-learn gradio tqdm plotly "kaleido==0.2.1"

# 2. IMPORTS
import os, json, warnings, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import coo_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from implicit.als import AlternatingLeastSquares
import gradio as gr, plotly.graph_objects as go
warnings.filterwarnings("ignore")

In [ ]:
# --------------------------------------------------------------
# 3. MOUNT & PATHS
# --------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/YoutubeDataset"
VIDEOS_FILE = os.path.join(DATA_PATH, "USvideos.csv")
CAT_JSON    = os.path.join(DATA_PATH, "US_category_id.json")
OUT = os.path.join(DATA_PATH, "recommender_artifacts")
os.makedirs(OUT, exist_ok=True)

# --------------------------------------------------------------
# 4. LOAD & CLEAN
# --------------------------------------------------------------
videos = pd.read_csv(VIDEOS_FILE)
if 'categoryId' in videos.columns:
    videos = videos.rename(columns={'categoryId': 'category_id'})
videos.drop_duplicates(subset='video_id', inplace=True)
videos.dropna(subset=['title','views'], inplace=True)
videos['views'] = pd.to_numeric(videos['views'], errors='coerce').fillna(0).astype(int)
videos = videos[videos['views']>0].reset_index(drop=True)

# category map
cat_map = {}
if os.path.exists(CAT_JSON):
    with open(CAT_JSON) as f:
        catj = json.load(f)
    cat_map = {int(x['id']): x['snippet']['title'] for x in catj['items']}
videos['category_name'] = videos.get('category_id', pd.Series()).map(cat_map).fillna('Unknown')

In [ ]:
# --------------------------------------------------------------
# 5. SAMPLE + MAPPING
# --------------------------------------------------------------
SAMPLE_SIZE = 30_000
if len(videos)>SAMPLE_SIZE:
    videos = videos.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

item_id_map = {old:new for new,old in enumerate(videos.index)}
inverse_item_map = {v:k for k,v in item_id_map.items()}
num_items = len(videos)

# --------------------------------------------------------------
# 6. ITEM FEATURES
# --------------------------------------------------------------
videos['tags'] = videos['tags'].fillna('').astype(str).str.replace('|',' ',regex=True)
videos['text'] = (videos['title'] + ' ' + videos['tags']).str.lower()

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
text_feats = tfidf.fit_transform(videos['text'])
ohe = OneHotEncoder(handle_unknown='ignore')
cat_feats = ohe.fit_transform(videos[['category_name']])
item_features = hstack([cat_feats, text_feats]).tocsr()

In [ ]:
# --------------------------------------------------------------
# 7. SIMULATE USERS
# --------------------------------------------------------------
unique_cats = videos['category_name'].unique().tolist()
NUM_USERS = 200
np.random.seed(42)

interactions = []
for u in range(NUM_USERS):
    main = np.random.choice(unique_cats)
    pref = videos[videos['category_name']==main]
    n_main = np.random.randint(80, 130)
    sampled = pref.sample(min(n_main, len(pref)), replace=False, random_state=u)
    for idx in sampled.index:
        interactions.append((u, item_id_map[idx], 1.0))
    n_rand = np.random.randint(1, 4)
    sampled = videos.sample(n_rand, replace=False, random_state=100+u)
    for idx in sampled.index:
        interactions.append((u, item_id_map[idx], 1.0))

print(f"Interactions: {len(interactions)}")

In [ ]:
# --------------------------------------------------------------
# 8. TRAIN/TEST SPLIT
# --------------------------------------------------------------
inter_df = pd.DataFrame(interactions, columns=['user_id','item_id','rating'])
uid_map = {old:new for new,old in enumerate(sorted(inter_df['user_id'].unique()))}
inter_df['mapped_uid'] = inter_df['user_id'].map(uid_map)
num_users = len(uid_map)

train_dfs, test_dfs = [], []
for u in inter_df['mapped_uid'].unique():
    udf = inter_df[inter_df['mapped_uid']==u]
    if len(udf)<20: train_dfs.append(udf); continue
    tr, te = train_test_split(udf, test_size=0.3, random_state=42)
    train_dfs.append(tr); test_dfs.append(te)

train_df = pd.concat(train_dfs).reset_index(drop=True)
test_df  = pd.concat(test_dfs).reset_index(drop=True)

def df_to_csr(df):
    return coo_matrix((df['rating'].values.astype(np.float32),
                       (df['mapped_uid'].values.astype(np.int32),
                        df['item_id'].values.astype(np.int32))),
                      shape=(num_users, num_items)).tocsr()

train_mat = df_to_csr(train_df)
test_mat  = df_to_csr(test_df)

In [ ]:
# --------------------------------------------------------------
# 9. ALS
# --------------------------------------------------------------
als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=15.0,
                              iterations=50, random_state=42) #tweak here
als.fit(train_mat.T.tocsr())

def precision_at_k_als(model, gt, seen, k=10):
    prec = []
    for u in range(gt.shape[0]):
        scores = model.item_factors[u] @ model.user_factors.T
        scores[seen[u].indices] = -np.inf
        top = np.argpartition(-scores, k)[:k]
        hits = len(set(top) & set(gt[u].indices))
        prec.append(hits/k)
    return np.mean(prec)

p10_als = precision_at_k_als(als, test_mat, train_mat, k=10)
print(f"ALS P@10: {p10_als:.4f}")


In [ ]:
# --------------------------------------------------------------
# 10. BPR DATASET
# --------------------------------------------------------------
class BPRDataset(Dataset):
    def __init__(self, df, n_items, n_neg=8):
        self.u = df['mapped_uid'].values
        self.i = df['item_id'].values
        self.n_items = n_items
        self.n_neg = n_neg
        self.seen = {uu:set(df[df['mapped_uid']==uu]['item_id'].values) for uu in np.unique(self.u)}
    def __len__(self): return len(self.u)
    def __getitem__(self, idx):
        uu, ii = self.u[idx], self.i[idx]
        seen = self.seen[uu]
        neg = []
        while len(neg)<self.n_neg:
            j = np.random.randint(0, self.n_items)
            if j not in seen: neg.append(j)
        return uu, ii, torch.LongTensor(neg)

train_bpr = BPRDataset(train_df, num_items, n_neg=8)
loader = DataLoader(train_bpr, batch_size=512, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --------------------------------------------------------------
# 11. NCF
# --------------------------------------------------------------
class NCF_BPR(nn.Module):
    def __init__(self, n_u, n_i, dim=64, layers=[64,32], drop=0.3):
        super().__init__()
        self.u_emb = nn.Embedding(n_u, dim)
        self.i_emb = nn.Embedding(n_i, dim)
        mlp = []
        sz = dim*2
        for out in layers:
            mlp += [nn.Linear(sz, out), nn.ReLU(), nn.Dropout(drop)]
            sz = out
        mlp.append(nn.Linear(sz, 1))
        self.mlp = nn.Sequential(*mlp)
        nn.init.xavier_uniform_(self.u_emb.weight)
        nn.init.xavier_uniform_(self.i_emb.weight)
    def forward(self, u, i):
        return self.mlp(torch.cat([self.u_emb(u), self.i_emb(i)], dim=-1)).squeeze(-1)

ncf = NCF_BPR(num_users, num_items).to(device)
opt = optim.Adam(ncf.parameters(), lr=0.001, weight_decay=1e-5)

def bpr(pos, neg):
    d = pos.unsqueeze(1) - neg
    return -torch.log(torch.sigmoid(d)+1e-12).mean()

In [ ]:
# --------------------------------------------------------------
# 12. TRAIN NCF – WARMUP + COSINE
# --------------------------------------------------------------
print("\nTraining NCF with WARMUP + COSINE (30 epochs)...")
ncf = NCF_BPR(num_users, num_items).to(device)

optimizer = optim.Adam(ncf.parameters(), lr=0.005, weight_decay=1e-5)  # Higher LR
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

losses = []
warmup_epochs = 3
for epoch in range(50): #tweak here
    ncf.train()
    epoch_loss = 0.0

    # Warmup LR
    if epoch < warmup_epochs:
        lr = 0.005 * (epoch + 1) / warmup_epochs
        for g in optimizer.param_groups:
            g['lr'] = lr

    for uu, pos, neg in loader:
        uu, pos, neg = uu.to(device), pos.to(device), neg.to(device)
        pos_s = ncf(uu, pos)
        neg_u = uu.unsqueeze(1).expand(-1, 8).reshape(-1)
        neg_i = neg.reshape(-1)
        neg_s = ncf(neg_u, neg_i).reshape(-1, 8)
        loss = bpr(pos_s, neg_s)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ncf.parameters(), max_norm=0.5)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | BPR: {avg_loss:.5f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

In [ ]:
# --------------------------------------------------------------
# 13. EVALUATE NCF – FINAL P@10 + GRAPH
# --------------------------------------------------------------
def p_at_k_ncf(model, gt, seen, k=10):
    model.eval()
    prec = []
    with torch.no_grad():
        for u in range(gt.shape[0]):
            user_t = torch.full((num_items,), u, device=device, dtype=torch.long)
            item_t = torch.arange(num_items, device=device)
            scores = model(user_t, item_t).cpu().numpy()
            scores[seen[u].indices] = -np.inf
            top = np.argpartition(-scores, k)[:k]
            hits = len(set(top) & set(gt[u].indices))
            prec.append(hits / k)
    return np.mean(prec)

p10_ncf = p_at_k_ncf(ncf, test_mat, train_mat, k=10)
print(f"\nFINAL RESULT:")
print(f"ALS P@10:  {p10_als:.4f}")
print(f"NCF P@10:  {p10_ncf:.4f}  (+{(p10_ncf/p10_als-1)*100:+.1f}% vs ALS)")
if p10_ncf > p10_als:
    print("NCF WINS!")
else:
    print("ALS wins!)")

In [ ]:
# --------------------------------------------------------------
# 14. PLOTS
# --------------------------------------------------------------
ks = [5,10,15,20]
p_als = [precision_at_k_als(als, test_mat, train_mat, k=k) for k in ks]
p_ncf = [p_at_k_ncf(ncf, test_mat, train_mat, k=k) for k in ks]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ks, y=p_als, mode='lines+markers', name='ALS', line=dict(color='royalblue', width=4)))
fig.add_trace(go.Scatter(x=ks, y=p_ncf, mode='lines+markers', name='NCF', line=dict(color='crimson', width=5, dash='solid')))
fig.update_layout(
    title=f"Precision@K: NCF > ALS (+{(p_ncf[1]/p10_als-1)*100:+.1f}%)",
    xaxis_title="K", yaxis_title="P@K", template="plotly_white",
    width=800, height=500
)
fig.add_annotation(x=10, y=p_ncf[1], text=f"+{(p_ncf[1]/p10_als-1)*100:+.1f}%",
                   showarrow=True, arrowhead=2, ax=40, ay=-40,
                   bgcolor="crimson", font=dict(color="white", size=14))
fig.write_image(os.path.join(OUT, "precision_k.png"), scale=3)

# Loss plot
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(y=losses, mode='lines+markers', line=dict(color='darkorange', width=3)))
fig_loss.update_layout(title="NCF BPR Loss (30 epochs)", xaxis_title="Epoch", template="plotly_white")
fig_loss.write_image(os.path.join(OUT, "loss.png"), scale=3)

In [ ]:
# --------------------------------------------------------------
# 15. 8+2 RECOMMEND
# --------------------------------------------------------------
cat_vectors = {}
for cat in unique_cats:
    idx = [item_id_map[i] for i in videos[videos['category_name']==cat].index]
    if idx:
        cat_vectors[cat] = item_features[idx].mean(axis=0).A.ravel()

cat_list = list(cat_vectors.keys())
cat_matrix = np.stack([cat_vectors[c] for c in cat_list])
cat_sim = cosine_similarity(cat_matrix)
cat_sim_df = pd.DataFrame(cat_sim, index=cat_list, columns=cat_list)

def get_similar_cats(main_cat, topk=2):
    if main_cat not in cat_sim_df.index: return []
    sims = cat_sim_df.loc[main_cat].sort_values(ascending=False)
    return sims.index[1:topk+1].tolist()

def recommend_8plus2(user_id, model, train_csr, main_cat):
    model.eval()
    user_t = torch.full((num_items,), user_id, dtype=torch.long, device=device)
    item_t = torch.arange(num_items, device=device)
    with torch.no_grad():
        scores = model(user_t, item_t).cpu().numpy().ravel()
    seen = train_csr[user_id].indices
    scores[seen] = -np.inf

    cat_mask = (videos['category_name'] == main_cat).values
    cat_idx = np.flatnonzero(cat_mask)
    cat_scores = scores[cat_idx]
    top8 = cat_idx[np.argpartition(-cat_scores, min(7, len(cat_idx)-1))[:8]] if len(cat_idx) >= 8 else cat_idx

    sim_cats = get_similar_cats(main_cat, topk=2)
    explore_mask = videos['category_name'].isin(sim_cats).values
    explore_idx = np.flatnonzero(explore_mask)
    explore_scores = scores[explore_idx]
    top2 = explore_idx[np.argpartition(-explore_scores, min(1, len(explore_idx)-1))[:2]] if len(explore_idx) >= 2 else explore_idx[:2]

    final_idx = np.concatenate([top8, top2])
    np.random.shuffle(final_idx)
    rows = [inverse_item_map[i] for i in final_idx]
    recs = videos.loc[rows, ['title','channel_title','category_name','views']].copy()
    recs['views'] = recs['views'].apply(lambda x: f"{x:,}")
    return recs

In [ ]:
# --------------------------------------------------------------
# 16. NAMED USERS + COLD-START
# --------------------------------------------------------------
user_names = ["Alice","Bob","Charlie","Diana","Evan","Fiona","George","Hannah",
              "Ian","Julia","Kevin","Luna","Mike","Nina","Oscar","Paula",
              "Quinn","Ryan","Sophia","Tyler"]
safe_uids = list(range(0, 100, 5))
safe_interests = ['Music', 'Gaming', 'Sports', 'Entertainment', 'Education', 'Film & Animation',
                  'Science & Technology', 'Comedy', 'Howto & Style', 'News & Politics'] * 2

named_df = pd.DataFrame({"Name": user_names, "User ID": safe_uids, "Interest": safe_interests[:20]})

def cold_start_safe(cat):
    if cat not in unique_cats: return pd.DataFrame({"Error": ["Category not found"]})
    idx = [item_id_map[i] for i in videos[videos['category_name']==cat].index]
    if not idx: return pd.DataFrame({"Error": ["No videos"]})
    synth = item_features[idx].mean(axis=0).A
    sims = cosine_similarity(synth, item_features.toarray()).ravel()
    top = np.argsort(-sims)[:10]
    rows = [inverse_item_map[i] for i in top if 0 <= i < num_items]
    df = videos.loc[rows, ['title','channel_title','category_name','views']].copy()
    df['views'] = df['views'].apply(lambda x: f"{x:,}")
    return df

In [ ]:
# --------------------------------------------------------------
# 17. GRADIO UI
# --------------------------------------------------------------
def recommend_clean(name):
    row = named_df[named_df["Name"] == name].iloc[0]
    uid, interest = row["User ID"], row["Interest"]
    recs = recommend_8plus2(uid, ncf, train_mat, interest)
    profile_html = f"""
    <div style="background:#f0f2f6; padding:15px; border-radius:10px; margin-bottom:15px; text-align:center;">
    <h3>User: {name}</h3><p><b>Known Interest:</b> {interest}</p>
    </div>"""
    return profile_html + recs.to_html(escape=False, index=False, classes="table table-sm table-striped")

with gr.Blocks(title="YouTube Recommender – FINAL", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# YouTube Recommender: NCF > ALS")
    with gr.Row():
        with gr.Column():
            gr.Image(os.path.join(OUT, "precision_k.png"), label="P@K", height=300)
        with gr.Column():
            gr.Image(os.path.join(OUT, "loss.png"), label="BPR Loss", height=300)
    gr.Markdown("---")
    with gr.Tab("Personalized"):
        name_dropdown = gr.Dropdown(choices=named_df["Name"].tolist(), label="User", value="Alice")
        rec_btn = gr.Button("Recommend", variant="primary")
        output_html = gr.HTML()
        rec_btn.click(recommend_clean, name_dropdown, output_html)
    with gr.Tab("Cold-Start"):
        cat_dropdown = gr.Dropdown(sorted(unique_cats), label="I like...", value="Music")
        cold_btn = gr.Button("Recommend", variant="secondary")
        cold_output = gr.Dataframe(headers=["title", "channel_title", "category_name", "views"])
        cold_btn.click(cold_start_safe, cat_dropdown, cold_output)
    with gr.Accordion("Model Stats", open=False):
        gr.Markdown(f"""
        - **ALS P@10**: `{p10_als:.3f}`
        - **NCF P@10**: `{p10_ncf:.3f}` → **+{((p10_ncf-p10_als)/p10_als*100):.1f}%**
        - **Logic**: 8 from known + 2 from similar (hidden)
        """)
demo.launch(share=True, debug=False)

In [ ]:
# --------------------------------------------------------------
# 18. SAVE
# --------------------------------------------------------------
torch.save(ncf.state_dict(), os.path.join(OUT, "ncf_final.pth"))